# `%catalog` — queue, commit, rollback

A worked example of the `%catalog` magic from `eea_datalakehouse.notebook.magics` — see
`docs/notebook-facade-for-data-scientists.md` for the design behind it.

**Before running this for real:** set `DREMIO_BASE_URL`, `DREMIO_TOKEN` (and `DREMIO_USERNAME`
if you'll queue `copy`/`move`) in the kernel environment — the same variables
`debugger/debug_run.py` uses. Install the extra this needs once: `pip install "EEADataLakehouse[notebook]"`.

Nothing below invents new vocabulary: every call after `%catalog` is a real
`CatalogSession` method (`src/eea_datalakehouse/catalog/session.py`) — this notebook is a tour of
that class, not a separate API.

In [ ]:
import eea_datalakehouse.notebook  # registers %catalog/%ingest — no %load_ext needed

`%load_ext eea_datalakehouse.notebook.magics` still works too, and is safe to run either
before or after the import above — whichever runs first registers the magics, the other is
a no-op (see `magics.load_ipython_extension`'s docstring).

## Queue a few steps

Each cell only **queues** an operation — nothing reaches Dremio yet. The same one
`CatalogSession` is reused across every `%catalog` cell in this kernel (see "Session
context lives in the Python process" in the design doc), so calls in different cells
accumulate on the same batch.

In [ ]:
%catalog copy("bwd.draft.raw_2026", "bwd.reference.water_temperature", overwrite=True)

%catalog tag(".water_temperature", ["reviewed", "2026"])

In [ ]:
%catalog tag("bwd.reference.water_temperature", ["reviewed", "2026"])

In [ ]:
%catalog set_meta("bwd.reference", tags=[
    {"tag_name": "owner", "tag_value": "bathing-water-team", "tag_title": "Owner"},
], overwrite=False)

## Commit — all-or-nothing

`commit()` runs every queued step in order. `retry=True` re-attempts a step that hits a
Dremio engine still warming up (`EngineStartingError`) a few times before giving up — any
other error rolls back immediately regardless. Either way the queue is empty afterwards,
whether this cell succeeds or raises.

In [ ]:
%catalog commit(retry=True)

`commit` with no parentheses also works, as a shorthand:

In [ ]:
%catalog commit

## What a failed commit looks like

If a step in the batch fails, `%catalog` prints a short message instead of a full
traceback (see the design doc's "Exceptions translated at the boundary") — for example,
queuing a step against a target that already exists without `overwrite=True`:

```
%catalog copy("bwd.draft.raw_2026", "bwd.reference.water_temperature")
%catalog commit
# -> catalog error: commit failed at step 1/1 (copy 'bwd.draft.raw_2026' -> ...):
#    target 'bwd.reference.water_temperature' already exists — pass overwrite=True ...
#    Everything before it was rolled back.
```

A batch mixing a reversible step with an `overwrite=True` step is explicit about what it
could *not* undo rather than pretending the rollback was clean — see `CatalogCommitError`'s
docstring, and "Must be all-or-nothing" in the design doc, for exactly which verbs that
applies to (`delete_folder`, and `copy`/`move` when `overwrite=True`).